# SignSight — EDA CAR35 V4

Dùng ZIP **CAR35 V4 đã chia sẵn**, không cần SupSign gốc, không chia lại train/val/test.
Tải dataset bằng **gdown**; không mount Drive. Notebook chạy độc lập, không clone repo.
Chọn CPU runtime. Dữ liệu chỉ nằm trong `/content`; lưu notebook không lưu dataset hay checkpoint.
Split: 25.886 train / 5.547 val / 5.547 test; 35 lớp. Có ảnh cùng cảnh giữa split và lớp hiếm ít mẫu holdout.
Đây là split phát triển, chưa phải test thực địa độc lập. Không so trực tiếp mAP với V3.


In [ ]:
import sys, subprocess, os
os.environ['YOLO_AUTOINSTALL'] = 'false'
os.environ['WANDB_MODE'] = 'disabled'
packages = ['Pillow>=10,<13', 'matplotlib>=3.8,<4', 'PyYAML>=6,<7', 'gdown==5.2.0']

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *packages])


## 1. Cấu hình

Upload ZIP CAR35 đã được cung cấp lên Drive, chia sẻ file cho phép tải bằng link, rồi dán link vào SOURCE_URL.
Giữ nguyên hai checksum để khóa đúng dữ liệu. Runtime/tài khoản mới chỉ cần notebook và link ZIP; gdown sẽ tải lại.
Không có DOWNLOAD=False để bỏ qua dataset ở runtime mới. Runtime cũ có dataset đã xác minh thì dùng lại.
Kết quả không tự lưu lên Drive. Tải ZIP kết quả trước khi runtime bị xóa.


In [ ]:
from pathlib import Path
import sys, os
WORK = Path('/content/signsight_car35')
WORK.mkdir(parents=True, exist_ok=True)
(WORK / '.ultralytics').mkdir(exist_ok=True)
os.environ['YOLO_CONFIG_DIR'] = str(WORK / '.ultralytics')
SOURCE_URL = ''  # Dán link file ZIP CAR35 trên Drive, không phải folder
DATASET_SHA256 = '78ff592aaf49d9539a92fd02334f6394a22ac1a592c32008a1416e4fe481e872'
EXPECTED_MANIFEST_SHA256 = '7d89760a0e83628a40895ee6a6367315aebb79b908a0101f50b66801479b1901'
DATASET_ROOT = WORK / 'dataset'
RUN_NAME = 'yolo11s_car35_v4_640'
RUN_DIR = WORK / 'runs' / RUN_NAME
EPOCHS = 80
IMAGE_SIZE = 640
BATCH = 8
DEVICE = '0'
RESUME = False  # Chỉ True khi RUN_DIR còn last.pt và training_request.json
RUN_HELDOUT_TEST = False  # Chỉ bật sau khi chốt cấu hình bằng validation
DOWNLOAD_RESULTS = True
assert RUN_NAME and Path(RUN_NAME).name == RUN_NAME and RUN_NAME not in ('.', '..')
print('Dataset local:', DATASET_ROOT)


In [ ]:
# Helpers được nhúng để notebook chạy độc lập sau khi upload.
import hashlib, importlib, json, shutil
TRAINING_FILES = {'training/__init__.py': '"""Offline datasets, experiments and notebook tooling; never imported by the API."""\n', 'training/data/__init__.py': '', 'training/experiments/__init__.py': '', 'training/data/common.py': "import hashlib, json\nfrom pathlib import Path\n\nCAR35 = [\n    'Speed_limit_40', 'Speed_limit_50', 'Speed_limit_60', 'Speed_limit_70', 'Speed_limit_80',\n    'No_entry', 'No_cars', 'No_cars_turn_left', 'No_cars_turn_right',\n    'No_turn_left', 'No_turn_right', 'No_U_turn', 'No_cars_U_turn',\n    'No_straight_ahead', 'No_turn_left_and_U_turn', 'No_turn_left_and_right',\n    'No_parking', 'No_stopping_and_parking', 'No_parking_odd_days', 'No_parking_even_days',\n    'Go_straight', 'Keep_right', 'Roundabout', 'One_way_road', 'Road_reserved_for_cars',\n    'Pedestrian_crossing', 'Children_crossing', 'Road_works', 'Slow_down',\n    'Dangerous_curve_left', 'Dangerous_curve_right',\n    'No_overtaking', 'End_no_overtaking', 'Give_way', 'End_all_prohibitions',\n]\nSPLITS = ('train', 'val', 'test')\nVERSION = 2\n\n\ndef digest_file(path):\n    with Path(path).open('rb') as stream:\n        digest = hashlib.sha256()\n        for block in iter(lambda: stream.read(1024*1024), b''): digest.update(block)\n        return digest.hexdigest()\n\n\ndef write_json(path, value):\n    Path(path).write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False), encoding='utf-8')\n\n\ndef read_json(path):\n    return json.loads(Path(path).read_text(encoding='utf-8'))\n\n\ndef recipe(classes=CAR35, seed=42, block_size=100, near_threshold=4, groups=None):\n    from training.compatibility import implementation_digest\n    group_digest = hashlib.sha256(json.dumps(groups, sort_keys=True).encode()).hexdigest()\n    return dict(version=VERSION, classes=list(classes), seed=seed, block_size=block_size,\n        near_threshold=near_threshold, groups_sha256=group_digest,\n        implementation_sha256=implementation_digest(),\n        reader_sha256=hashlib.sha256(Path(__file__).with_name('audit.py').read_text(encoding='utf-8').encode()).hexdigest())\n", 'training/data/downloads.py': '"""Download a shared Drive ZIP to local Colab storage, without account cookies."""\nimport hashlib\nimport json\nfrom pathlib import Path\nimport re\nfrom urllib.parse import parse_qs, urlparse\nimport zipfile\n\n\ndef drive_file_id(source):\n    source = str(source).strip()\n    if not source or \'YOUR_\' in source or \'PASTE_\' in source:\n        raise ValueError(\'Paste a shared Google Drive file URL or file ID\')\n    if \'://\' in source:\n        parsed = urlparse(source)\n        if parsed.scheme != \'https\' or parsed.hostname != \'drive.google.com\' or parsed.username:\n            raise ValueError(\'Use a Google Drive file link, not a folder or another website\')\n        match = re.fullmatch(r\'/file/d/([A-Za-z0-9_-]+)(?:/.*)?\', parsed.path)\n        if match:\n            source = match[1]\n        elif parsed.path in (\'/uc\', \'/open\'):\n            source = parse_qs(parsed.query).get(\'id\', [\'\'])[0]\n        else:\n            raise ValueError(\'Use a Google Drive file link, not a folder\')\n    if not re.fullmatch(r\'[A-Za-z0-9_-]{10,}\', source):\n        raise ValueError(\'Invalid Google Drive file ID\')\n    return source\n\n\ndef download_drive_zip(source, target, *, expected_sha256=\'\', downloader=None):\n    """Publish only a complete ZIP; source receipts prevent resuming another file.\n\n    gdown owns its resumable temporary files. This helper keeps those on network\n    failure and never overwrites a pre-existing target from an unknown source.\n    An authoritative SHA-256 can adopt an already downloaded matching ZIP.\n    """\n    file_id = drive_file_id(source)\n    checksum = str(expected_sha256).strip().lower()\n    if checksum and not re.fullmatch(\'[a-f0-9]{64}\', checksum):\n        raise ValueError(\'Expected SHA-256 must have 64 hexadecimal characters\')\n    target = Path(target).resolve()\n    receipt = target.with_name(target.name+\'.source.json\')\n    staging = target.with_name(target.name+\'.download\')\n\n    def validate(path):\n        if not path.is_file() or not zipfile.is_zipfile(path):\n            raise ValueError(\'Download is not a ZIP. Check Drive sharing, quota and the file link.\')\n        if checksum:\n            with path.open(\'rb\') as stream:\n                actual = hashlib.file_digest(stream, \'sha256\').hexdigest()\n            if actual != checksum: raise ValueError(\'Downloaded ZIP SHA-256 mismatch\')\n\n    known = receipt.exists()\n    if known:\n        try:\n            same_source = json.loads(receipt.read_text())[\'file_id\'] == file_id\n        except (ValueError, KeyError, TypeError):\n            same_source = False\n        if not same_source: raise FileExistsError(\'Local download belongs to another Drive file; choose a new path\')\n    elif target.exists():\n        if not checksum: raise FileExistsError(\'Existing ZIP has no source receipt; supply its SHA-256 or choose a new path\')\n        validate(target)\n    elif staging.exists() or any(target.parent.glob(staging.name+\'*.tmp\')):\n        raise FileExistsError(\'Unknown partial download exists; choose a new local path\')\n    target.parent.mkdir(parents=True, exist_ok=True)\n    if not known:\n        with receipt.open(\'x\', encoding=\'utf-8\') as stream:\n            json.dump({\'file_id\': file_id}, stream)\n    if target.exists():\n        validate(target)\n        return target\n    if staging.exists():\n        try:\n            validate(staging)\n        except ValueError:\n            staging.unlink()  # Only the staging file identified by our matching receipt.\n    if not staging.exists():\n        if downloader is None:\n            import gdown\n            downloader = gdown.download\n        try:\n            result = downloader(id=file_id, output=str(staging), quiet=False,\n                                resume=True, use_cookies=False)\n        except Exception as error:\n            raise RuntimeError(\'Drive download failed. Check public-link access/quota, then rerun this cell \'\n                               \'to resume in the same runtime.\') from error\n        if result is None:\n            raise RuntimeError(\'Drive returned no download. Check public-link access and download quota.\')\n    try:\n        validate(staging)\n    except ValueError:\n        staging.unlink(missing_ok=True)\n        raise\n    if target.exists(): raise FileExistsError(\'Download target appeared; it was not overwritten\')\n    staging.rename(target)\n    return target\n', 'training/data/car35_bundle.py': '"""Restore and verify a prepared CAR35 V4 ZIP without source data or re-splitting."""\nfrom collections import Counter\nfrom concurrent.futures import ThreadPoolExecutor\nimport hashlib\nimport json\nimport math\nfrom pathlib import Path, PurePosixPath\nimport re\nimport shutil\nimport tempfile\nimport zipfile\n\nimport yaml\nfrom training.data.common import CAR35, SPLITS, digest_file\n\n\ndef verify_car35(root, *, expected_manifest_sha256=\'\', workers=8):\n    root=Path(root).resolve();manifest_file=root/\'manifest.json\'\n    if expected_manifest_sha256 and digest_file(manifest_file)!=expected_manifest_sha256:\n        raise ValueError(\'CAR35 manifest SHA-256 mismatch\')\n    manifest=json.loads(manifest_file.read_text(encoding=\'utf-8\'))\n    if manifest.get(\'kind\')!=\'car35_folder_development_v4\' or manifest.get(\'names\')!=CAR35:\n        raise ValueError(\'Expected prepared CAR35 V4 taxonomy, not raw SupSign or a V3 cache\')\n    config=yaml.safe_load((root/\'data.yaml\').read_text(encoding=\'utf-8\'))\n    if config!=dict(train=\'images/train\',val=\'images/val\',test=\'images/test\',names=CAR35):\n        raise ValueError(\'Unsafe or changed CAR35 data.yaml\')\n    if manifest.get(\'independent_field_test\') is not False:\n        raise ValueError(\'This development bundle is not an independent field test\')\n    if type(workers) is not int or not 1<=workers<=32:raise ValueError(\'Invalid worker count\')\n    images,labels,hashes,groups=set(),set(),set(),{}\n    counts={s:Counter() for s in SPLITS};sizes=Counter()\n    def check(row):\n        split=row[\'split\'];image,label=(root/row[k] for k in (\'image\',\'label\'));sha=row[\'sha256\']\n        if (split not in SPLITS or not re.fullmatch(\'[0-9a-f]{64}\',sha)\n            or image.parent!=root/\'images\'/split or image.stem!=sha\n            or image.suffix.lower() not in {\'.png\',\'.jpg\',\'.jpeg\',\'.webp\',\'.bmp\',\'.tif\',\'.tiff\'}\n            or label!=root/\'labels\'/split/(sha+\'.txt\')\n            or not image.resolve().is_relative_to(root) or not label.resolve().is_relative_to(root)):\n            raise ValueError(\'Unsafe manifest image/label path\')\n        content=label.read_bytes()\n        if digest_file(image)!=sha or hashlib.sha256(content).hexdigest()!=row[\'label_sha256\']:\n            raise ValueError(\'CAR35 image/label hash changed\')\n        current=Counter()\n        for line in content.decode(\'utf-8\').splitlines():\n            if not line.strip():continue\n            fields=line.split()\n            if len(fields)!=5:raise ValueError(\'Invalid detection label\')\n            c=int(fields[0]);x,y,w,h=map(float,fields[1:])\n            if (not 0<=c<35 or not all(math.isfinite(v) for v in (x,y,w,h)) or min(w,h)<=0\n                or min(x-w/2,y-h/2)<-1e-6 or max(x+w/2,y+h/2)>1+1e-6):\n                raise ValueError(\'Invalid CAR35 class/bbox\')\n            current[c]+=1\n        return row,image,label,current\n    with ThreadPoolExecutor(max_workers=workers) as pool:\n        for row,image,label,current in pool.map(check,manifest[\'images\']):\n            if row[\'sha256\'] in hashes or (row[\'group\'] in groups and groups[row[\'group\']]!=row[\'split\']):\n                raise ValueError(\'Duplicate hash or cross-split CAR35 group\')\n            images.add(image);labels.add(label);hashes.add(row[\'sha256\']);groups[row[\'group\']]=row[\'split\']\n            sizes[row[\'split\']]+=1;counts[row[\'split\']].update(current)\n    actual_labels=set((root/\'labels\').rglob(\'*.*\'))\n    allowed_caches={root/\'labels\'/(s+suffix) for s in SPLITS for suffix in (\'.cache\',\'.cache.npy\')}\n    for cache in actual_labels & allowed_caches:\n        if cache.is_symlink() or not cache.is_file() or not cache.resolve().is_relative_to(root):\n            raise ValueError(\'Unsafe YOLO label cache path\')\n    if images!=set((root/\'images\').rglob(\'*.*\')) or labels!=actual_labels-allowed_caches:\n        raise ValueError(\'CAR35 image/label inventory changed\')\n    if set(sizes)!=set(SPLITS):raise ValueError(\'Empty CAR35 split\')\n    for split in SPLITS:\n        recorded={int(k):v for k,v in manifest[\'class_counts\'][split].items() if v}\n        if recorded!=dict(counts[split]):raise ValueError(\'CAR35 class counts changed\')\n    if any(not counts[\'train\'][c] for c in range(35)):raise ValueError(\'Missing CAR35 train class\')\n    return dict(images=len(images),boxes=sum(sum(c.values()) for c in counts.values()),names=list(CAR35),\n                split_images=dict(sizes),split_groups=dict(Counter(groups.values())),\n                class_counts={s:dict(counts[s]) for s in SPLITS},groups=len(groups),\n                manifest_sha256=digest_file(manifest_file),independent_field_test=False,\n                limitations=manifest.get(\'limitations\',\'Development split; scene overlap may remain.\'))\n\n\ndef restore_car35_bundle(archive,output,*,expected_sha256,expected_manifest_sha256=\'\'):\n    archive,output=Path(archive).resolve(),Path(output).resolve()\n    if output.exists():raise FileExistsError(output)\n    if not re.fullmatch(\'[0-9a-fA-F]{64}\',expected_sha256 or \'\'):\n        raise ValueError(\'CAR35 archive SHA-256 is required\')\n    if digest_file(archive)!=expected_sha256.lower():raise ValueError(\'CAR35 ZIP checksum mismatch\')\n    with zipfile.ZipFile(archive) as source:\n        entries=source.infolist();seen=set()\n        for member in entries:\n            name=member.filename;path=PurePosixPath(name)\n            if (not name or path.is_absolute() or \'..\' in path.parts or \'\\\\\' in name or \':\' in name\n                or name.casefold() in seen or (member.external_attr>>16)&0o170000==0o120000\n                or member.flag_bits&1):raise ValueError(\'Unsafe or duplicate CAR35 ZIP member\')\n            seen.add(name.casefold())\n        if not {\'data.yaml\',\'manifest.json\'}<=seen:raise ValueError(\'ZIP is not a flat prepared CAR35 bundle\')\n        required=sum(m.file_size for m in entries)+1024**3\n        output.parent.mkdir(parents=True,exist_ok=True)\n        if shutil.disk_usage(output.parent).free<required:raise ValueError(\'Not enough disk space to restore CAR35\')\n        with tempfile.TemporaryDirectory(prefix=\'.car35-restore-\',dir=output.parent) as temporary:\n            stage=Path(temporary)/\'dataset\';stage.mkdir()\n            for member in entries:\n                if member.is_dir():continue\n                target=stage/member.filename;target.parent.mkdir(parents=True,exist_ok=True)\n                with source.open(member) as stream,target.open(\'wb\') as dest:shutil.copyfileobj(stream,dest)\n            info=verify_car35(stage,expected_manifest_sha256=expected_manifest_sha256)\n            stage.rename(output)\n    return info\n', 'training/experiments/car35.py': '"""Train/export a prepared CAR35 V4 dataset; never changes application serving."""\nimport json\nimport math\nfrom pathlib import Path\nimport zipfile\n\nfrom training.data.car35_bundle import verify_car35\nfrom training.data.common import digest_file, read_json, write_json\n\n\ndef train_car35(root,run_dir,*,epochs=80,imgsz=640,batch=8,device=\'0\',resume=False,yolo_factory=None):\n    root,run_dir=Path(root).resolve(),Path(run_dir).resolve()\n    if run_dir.is_relative_to(root) or root.is_relative_to(run_dir):raise ValueError(\'Run must be outside dataset\')\n    if epochs<1 or imgsz<32 or batch<1:raise ValueError(\'Invalid training settings\')\n    dataset=verify_car35(root)\n    settings=dict(model=\'yolo11s.pt\',epochs=epochs,imgsz=imgsz,batch=batch,seed=42)\n    request=run_dir/\'training_request.json\'\n    if resume:\n        if not request.is_file() or not (run_dir/\'weights/last.pt\').is_file():\n            raise ValueError(\'Resume requires last.pt and training_request.json in this runtime\')\n        previous=read_json(request)\n        if previous[\'dataset\'][\'manifest_sha256\']!=dataset[\'manifest_sha256\']:raise ValueError(\'Resume dataset changed\')\n        if previous[\'settings\']!=settings:raise ValueError(\'Resume settings changed\')\n        weights=run_dir/\'weights/last.pt\'\n    else:\n        if run_dir.exists():raise FileExistsError(run_dir)\n        weights=\'yolo11s.pt\'\n    if yolo_factory is None:\n        from ultralytics import YOLO\n        yolo_factory=YOLO\n    detector=yolo_factory(str(weights))\n    if resume:\n        names=detector.names;names=[names[i] for i in range(len(names))] if isinstance(names,dict) else list(names)\n        if detector.task!=\'detect\' or names!=dataset[\'names\']:raise ValueError(\'Resume checkpoint taxonomy differs\')\n    if not resume:\n        run_dir.mkdir(parents=True)\n        write_json(request,dict(dataset=dataset,settings=settings,test_evaluated=False))\n    options=dict(data=str(root/\'data.yaml\'),epochs=epochs,imgsz=imgsz,batch=batch,device=device,\n        project=str(run_dir.parent),name=run_dir.name,exist_ok=True,split=\'val\',workers=2,\n        seed=42,deterministic=True,cache=False,amp=True,patience=20,save=True,save_period=5,\n        fliplr=0.,flipud=0.,degrees=0.,shear=0.,perspective=0.,mixup=0.,mosaic=.5,\n        close_mosaic=min(10,epochs),plots=True)\n    if resume:options[\'resume\']=True\n    detector.train(**options)\n    best=run_dir/\'weights/best.pt\'\n    if not best.is_file():raise ValueError(\'Training did not produce best.pt\')\n    return best\n\n\ndef evaluate_car35(checkpoint,root,output,*,split=\'val\',allow_test=False,imgsz=640,batch=8,device=\'0\',yolo_factory=None):\n    if split not in (\'val\',\'test\') or split==\'test\' and not allow_test:raise ValueError(\'Held-out test requires explicit opt-in\')\n    root,output,checkpoint=Path(root).resolve(),Path(output).resolve(),Path(checkpoint).resolve()\n    if output.exists():raise FileExistsError(output)\n    if output.is_relative_to(root) or root.is_relative_to(output):raise ValueError(\'Evaluation must be outside dataset\')\n    dataset=verify_car35(root)\n    if yolo_factory is None:\n        from ultralytics import YOLO\n        yolo_factory=YOLO\n    detector=yolo_factory(str(checkpoint));names=detector.names\n    names=[names[i] for i in range(len(names))] if isinstance(names,dict) else list(names)\n    if detector.task!=\'detect\' or names!=dataset[\'names\']:raise ValueError(\'Evaluation checkpoint taxonomy differs\')\n    output.mkdir(parents=True)\n    result=detector.val(data=str(root/\'data.yaml\'),split=split,imgsz=imgsz,batch=batch,device=device,\n                        project=str(output.parent),name=output.name,exist_ok=True,plots=True)\n    report=dict(dataset=dataset,split=split,split_images=dataset[\'split_images\'][split],\n                checkpoint_sha256=digest_file(checkpoint),independent_field_test=False,\n                metrics={k:float(v) if math.isfinite(float(v)) else None for k,v in result.results_dict.items()})\n    box=getattr(result,\'box\',None)\n    if box is not None:\n        report[\'per_class\']=[dict(class_id=int(c),name=names[int(c)],precision=float(box.p[j]),\n            recall=float(box.r[j]),map50=float(box.ap50[j]),map50_95=float(box.ap[j])) for j,c in enumerate(box.ap_class_index)]\n    write_json(output/\'evaluation_report.json\',report)\n    return report\n\n\ndef export_car35_run(run_dir,root,target):\n    run_dir,root,target=Path(run_dir).resolve(),Path(root).resolve(),Path(target).resolve()\n    if target.exists():raise FileExistsError(target)\n    if target.is_relative_to(run_dir) or target.is_relative_to(root):raise ValueError(\'Export must be outside run/dataset\')\n    if not any((run_dir/\'weights\'/n).is_file() for n in (\'best.pt\',\'last.pt\')):raise ValueError(\'No checkpoint to export yet\')\n    dataset=verify_car35(root)\n    request=read_json(run_dir/\'training_request.json\')\n    if request[\'dataset\'][\'manifest_sha256\']!=dataset[\'manifest_sha256\']:raise ValueError(\'Export dataset changed\')\n    handoff=dict(schema_version=1,dataset=dataset,settings=request[\'settings\'],independent_field_test=False,\n        checkpoints={n:digest_file(run_dir/\'weights\'/n) for n in (\'best.pt\',\'last.pt\') if (run_dir/\'weights\'/n).is_file()},\n        limitations=\'Development CAR35 V4; scene overlap and rare holdout classes remain. Review before activation.\')\n    target.parent.mkdir(parents=True,exist_ok=True)\n    with zipfile.ZipFile(target,\'x\',compression=zipfile.ZIP_DEFLATED,allowZip64=True) as archive:\n        for path in sorted(run_dir.rglob(\'*\')):\n            if path.is_file() and not path.is_symlink():archive.write(path,\'run/\'+path.relative_to(run_dir).as_posix())\n        for name in (\'manifest.json\',\'data.yaml\',\'dataset_report.json\',\'class_counts.csv\',\'README.md\'):\n            if (root/name).is_file():archive.write(root/name,\'dataset/\'+name)\n        archive.writestr(\'handoff.json\',json.dumps(handoff,ensure_ascii=False,indent=2))\n    return target\n'}
PIPELINE_ID = hashlib.sha256(json.dumps(TRAINING_FILES, sort_keys=True).encode()).hexdigest()
CODE_ROOT = WORK / 'pipeline' / PIPELINE_ID
for relative, source in TRAINING_FILES.items():
    target = CODE_ROOT / relative
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(source, encoding='utf-8')
for name in list(sys.modules):
    if name in ('training', 'training.data', 'training.experiments') or name in (
        'training.data.common', 'training.data.downloads', 'training.data.car35_bundle', 'training.experiments.car35'):
        del sys.modules[name]
sys.path.insert(0, str(CODE_ROOT))
importlib.invalidate_caches()
from training.data.downloads import download_drive_zip
from training.data.car35_bundle import restore_car35_bundle, verify_car35
from training.experiments.car35 import train_car35, evaluate_car35, export_car35_run
print('Pipeline:', PIPELINE_ID[:16])


In [ ]:
if DATASET_ROOT.exists():
    dataset_info = verify_car35(DATASET_ROOT, expected_manifest_sha256=EXPECTED_MANIFEST_SHA256)
else:
    assert SOURCE_URL.strip(), 'Điền SOURCE_URL bằng link file ZIP CAR35 trên Drive.'
    archive = download_drive_zip(SOURCE_URL, WORK / 'downloads' / 'car35_v4.zip', expected_sha256=DATASET_SHA256)
    dataset_info = restore_car35_bundle(archive, DATASET_ROOT,
        expected_sha256=DATASET_SHA256, expected_manifest_sha256=EXPECTED_MANIFEST_SHA256)
manifest = json.loads((DATASET_ROOT / 'manifest.json').read_text(encoding='utf-8'))
print(f"CAR35: {dataset_info['images']:,} images | {dataset_info['boxes']:,} boxes | 35 classes")
for split in ('train', 'val', 'test'):
    print(split, ':', dataset_info['split_images'][split], 'images')
print('Giữ nguyên split. Test thực địa độc lập:', dataset_info['independent_field_test'])


## 2. EDA đúng bộ CAR35 đã đóng gói

Không đọc lại SupSign gốc hay đổi split. Biểu đồ dùng số bbox đã đối soát với nhãn thực tế.
ZIP EDA chứa manifest, bảng lớp, biểu đồ và báo cáo/ảnh preview đi kèm dataset.
Xem `report.html`: có ảnh cùng cảnh ở nhiều split; hai lớp hết cấm chỉ có một bbox test/lớp.


In [ ]:
import csv
import matplotlib.pyplot as plt
EDA_ROOT = WORK / 'eda'; EDA_ROOT.mkdir(exist_ok=True)
rows = [dict(class_id=i, name=name, **{s:dataset_info['class_counts'][s].get(i, 0) for s in ('train','val','test')})
        for i,name in enumerate(dataset_info['names'])]
with (EDA_ROOT / 'class_counts.csv').open('w', newline='', encoding='utf-8-sig') as f:
    writer = csv.DictWriter(f, fieldnames=list(rows[0])); writer.writeheader(); writer.writerows(rows)
fig, ax = plt.subplots(figsize=(12, 12)); left = [0]*35
for split in ('train','val','test'):
    values = [r[split] for r in rows]
    ax.barh(range(35), values, left=left, label=split)
    left = [a+b for a,b in zip(left,values)]
ax.set_yticks(range(35), dataset_info['names'], fontsize=8); ax.legend(); ax.set_title('CAR35 V4 boxes by split')
fig.tight_layout(); fig.savefig(EDA_ROOT / 'class_distribution.png', dpi=150); plt.close(fig)
(EDA_ROOT / 'summary.json').write_text(json.dumps(dataset_info, ensure_ascii=False, indent=2), encoding='utf-8')
for name in ('manifest.json','data.yaml','report.html','dataset_report.json','README.md'):
    if (DATASET_ROOT / name).is_file(): shutil.copy2(DATASET_ROOT / name, EDA_ROOT / name)
if (DATASET_ROOT / 'previews').is_dir(): shutil.copytree(DATASET_ROOT / 'previews', EDA_ROOT / 'previews', dirs_exist_ok=True)
print('EDA saved:', EDA_ROOT)


In [ ]:
from datetime import datetime, timezone
name = 'car35_v4_eda_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
RESULT_ZIP = Path(shutil.make_archive(str(WORK / name), 'zip', EDA_ROOT))
print('EDA ZIP:', RESULT_ZIP)
if DOWNLOAD_RESULTS:
    from google.colab import files
    files.download(str(RESULT_ZIP))
